# OBJECTIVE

# Refactor the delivery-time prediction code built in P02/P03 from scattered notebook cells into a proper Python package (delivery/) with separate modules — data.py, features.py, model.py, and validate.py — tied together with an __init__.py. Import and use this package like a library to train and save a model, then run the full pipeline from the command line using standalone scripts (train.py and predict.py), without relying on a notebook.

# Tasks:

# T1: Add a function average_speed_kmph(distance_km, delivery_min) to the package.

# T2: Add a delivery/validate.py module with a function that rejects an impossible/unrealistic order.

# T3: Write a second command-line script, predict.py, that loads the saved model and predicts delivery time for a new order.

# Create Folders

In [2]:
#`delivery/` will hold our package files, `data/` will hold the CSV.

In [3]:
import os
os.makedirs("delivery", exist_ok=True)
os.makedirs("data", exist_ok=True)
print("Folders ready")

Folders ready


# Step 1: data.py --- loads the data

In [4]:
%%writefile delivery/data.py
import pandas as pd
import numpy as np
import os

def load_data():
    path = "data/delivery_times.csv"

    if not os.path.exists(path):
        np.random.seed(42)
        n = 600
        distance_km = np.random.uniform(0.5, 12, n)
        prep_time_min = np.random.uniform(5, 30, n)
        traffic_level = np.random.randint(1, 4, n)
        rain = np.random.randint(0, 2, n)
        noise = np.random.normal(0, 2, n)

        delivery_min = 6 + 3 * distance_km + 0.6 * prep_time_min + 4 * traffic_level + 5 * rain + noise

        df = pd.DataFrame({
            "distance_km": distance_km,
            "prep_time_min": prep_time_min,
            "traffic_level": traffic_level,
            "rain": rain,
            "delivery_min": delivery_min
        })
        df.to_csv(path, index=False)

    return pd.read_csv(path)

Writing delivery/data.py


# Step 2: features.py --- splits X and y (also has T1)

# Splits the data into inputs (X) and the target to predict (y).
# average_speed_kmph is Task T1.

In [5]:
%%writefile delivery/features.py
def get_features_and_target(df):
    X = df[["distance_km", "prep_time_min", "traffic_level", "rain"]]
    y = df["delivery_min"]
    return X, y

# T1
def average_speed_kmph(distance_km, delivery_min):
    hours = delivery_min / 60
    return distance_km / hours

Writing delivery/features.py


# Step 3: model.py --- trains, checks, and saves the model

# Trains a LinearRegression model, prints its test error (MAE), and saves it to a file so predict.py can reuse it later.

In [6]:
%%writefile delivery/model.py
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

def train_and_save_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    print("Test MAE:", round(mae, 2))

    joblib.dump(model, "delivery_model.joblib")
    return model

Writing delivery/model.py


# Step 4: validate.py --- T2, rejects a bad order
# Checks that an order's values are realistic before we predict on it.

In [7]:
%%writefile delivery/validate.py
# T2
def is_valid_order(distance_km, prep_time_min, traffic_level, rain):
    if distance_km <= 0:
        return False
    if prep_time_min <= 0:
        return False
    if traffic_level not in [1, 2, 3]:
        return False
    if rain not in [0, 1]:
        return False
    return True

Writing delivery/validate.py


# Step 5: __init__.py --- makes delivery a package

# Built last so it can import functions from all the other files. This is what makes `from delivery import ...` work.

In [8]:
%%writefile delivery/__init__.py
from .data import load_data
from .features import get_features_and_target, average_speed_kmph
from .model import train_and_save_model
from .validate import is_valid_order

Writing delivery/__init__.py


# Step 6: Import and use our own package
# Proves the package works --- import it like a library and train the model.

In [9]:
from delivery import load_data, get_features_and_target, train_and_save_model

df = load_data()
X, y = get_features_and_target(df)
model = train_and_save_model(X, y)

Test MAE: 1.4


# Step 7: train.py  script

In [10]:
%%writefile train.py
from delivery import load_data, get_features_and_target, train_and_save_model

df = load_data()
X, y = get_features_and_target(df)
model = train_and_save_model(X, y)
print("Training done")

Writing train.py


# Step 8: Run train.py like a real program
# Runs the script as a terminal command, no notebook involved.

In [11]:
!python train.py

Test MAE: 1.4
Training done


# T3: predict.py --- second script
# Loads the saved model and predicts delivery time for one new order.

In [12]:
%%writefile predict.py
import sys
import joblib
import pandas as pd
from delivery import is_valid_order

distance_km = float(sys.argv[1])
prep_time_min = float(sys.argv[2])
traffic_level = int(sys.argv[3])
rain = int(sys.argv[4])

if not is_valid_order(distance_km, prep_time_min, traffic_level, rain):
    print("Invalid order")
else:
    model = joblib.load("delivery_model.joblib")
    order = pd.DataFrame([[distance_km, prep_time_min, traffic_level, rain]],
                          columns=["distance_km", "prep_time_min", "traffic_level", "rain"])
    prediction = model.predict(order)[0]
    print("Predicted delivery time:", round(prediction, 1), "minutes")

Writing predict.py


# Test predict.py --- a normal order

In [13]:
!python predict.py 5.0 15 2 0

Predicted delivery time: 37.9 minutes


# Test predict.py --- a bad order (T2 check)

In [14]:
!python predict.py -3 15 2 0

Invalid order


# Test T1: average_speed_kmph

In [15]:
from delivery import average_speed_kmph

speed = average_speed_kmph(distance_km=5.0, delivery_min=40.4)
print("Average speed (km/h):", round(speed, 2))

Average speed (km/h): 7.43


# Summary


- data.py --- loads the delivery data
- features.py --- splits data into X and y
- model.py --- trains, checks, and saves the model
- validate.py --- rejects a bad order
- __init__.py --- makes the folder a package

# To Do: Package the classification workflow

Using the same pattern from this lab (data.py -> features.py -> model.py ->
validate.py -> __init__.py), turn the classification code from Lab 3
(Breast Cancer dataset) into its own package. This time, go a step further
than just training one fixed model.

T1 --- model.py should not train just one model. Use cross-validation to
compare LogisticRegression, DecisionTreeClassifier, and
RandomForestClassifier, and automatically save whichever one scores best
(instead of hardcoding the winner yourself).

T2 --- validate.py should check that at least 3 of the input measurements
fall within a realistic range (e.g. radius_mean and area_mean can't be
negative or absurdly large) --- not just "is this a number".

T3 --- predict.py should print both the prediction (malignant/benign) AND
the model's confidence for that prediction (use predict_proba).

T4 --- add a metrics.py module with one function that prints a
confusion matrix and classification report for the saved model, so anyone
can check its performance without retraining.

In [1]:
import os
os.makedirs("clf_delivery", exist_ok=True)   # classification package
os.makedirs("clf_data", exist_ok=True)       # CSV yahan
print("Folders ready")

Folders ready


In [16]:
%%writefile clf_delivery/data.py
import pandas as pd
from sklearn.datasets import load_breast_cancer

def load_data():
    data = load_breast_cancer()
    df = pd.DataFrame(data.data, columns=data.feature_names)
    df["target"] = data.target          # 0 = malignant, 1 = benign
    return df

Writing clf_delivery/data.py


In [17]:
%%writefile clf_delivery/features.py
def get_features_and_target(df):
    feature_cols = [c for c in df.columns if c != "target"]
    X = df[feature_cols]
    y = df["target"]
    return X, y

Writing clf_delivery/features.py


In [18]:
%%writefile clf_delivery/model.py
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score
import joblib

def _candidates():
    return {
        "LogisticRegression": LogisticRegression(max_iter=5000),
        "DecisionTree": DecisionTreeClassifier(max_depth=4, random_state=42),
        "RandomForest": RandomForestClassifier(n_estimators=50, random_state=42),
    }

def train_best_and_save(X, y):
    # chose better model than 5 fold cv
    scores = {}
    for name, model in _candidates().items():
        cv = cross_val_score(model, X, y, cv=5, scoring="accuracy")
        scores[name] = cv.mean()
        print(f"{name:20s} CV accuracy = {cv.mean():.4f}")

    best_name = max(scores, key=scores.get)
    print("\nBest model:", best_name)

    # Fit best model on full training data
    best_model = _candidates()[best_name]
    best_model.fit(X, y)

    joblib.dump(best_model, "clf_model.joblib")
    joblib.dump(best_name, "clf_model_name.joblib")
    return best_model, best_name

Writing clf_delivery/model.py


In [30]:
%%writefile clf_delivery/validate.py
def is_valid_order(radius_mean, area_mean, texture_mean, perimeter_mean):
    """
    Breast cancer realistic ranges:
      radius_mean    : 6 - 30
      area_mean      : 140 - 2600
      texture_mean   : 9 - 40
      perimeter_mean : 40 - 200
    Negative ya zero values hamesha reject.
    """
    # Hard reject: koi bhi measurement <= 0 ho toh invalid
    if radius_mean <= 0 or area_mean <= 0 or texture_mean <= 0 or perimeter_mean <= 0:
        print("Validation: negative/zero value found — rejected")
        return False

    checks = [
        6   <= radius_mean    <= 30,
        140 <= area_mean      <= 2600,
        9   <= texture_mean   <= 40,
        40  <= perimeter_mean <= 200,
    ]
    passed = sum(checks)
    print(f"Validation: {passed}/4 measurements in realistic range")
    return passed >= 3

Overwriting clf_delivery/validate.py


In [20]:
%%writefile clf_delivery/metrics.py
from sklearn.metrics import confusion_matrix, classification_report
import joblib

def evaluate_saved_model(X, y):
    """Saved model load karke confusion matrix + classification report print karta hai."""
    model = joblib.load("clf_model.joblib")

    preds = model.predict(X)

    print("Confusion matrix:")
    print(confusion_matrix(y, preds))
    print()
    print("Classification report:")
    print(classification_report(y, preds, target_names=["malignant", "benign"]))

Writing clf_delivery/metrics.py


In [21]:
%%writefile clf_delivery/__init__.py
from .data import load_data
from .features import get_features_and_target
from .model import train_best_and_save
from .validate import is_valid_order
from .metrics import evaluate_saved_model

Writing clf_delivery/__init__.py


In [22]:
from clf_delivery import (
    load_data, get_features_and_target, train_best_and_save,
    is_valid_order, evaluate_saved_model
)

df = load_data()
X, y = get_features_and_target(df)
print("Shape:", X.shape)

best_model, best_name = train_best_and_save(X, y)
print("\nSaved model:", best_name)

print("\n--- Evaluating saved model on full data ---")
evaluate_saved_model(X, y)

Shape: (569, 30)
LogisticRegression   CV accuracy = 0.9508
DecisionTree         CV accuracy = 0.9209
RandomForest         CV accuracy = 0.9543

Best model: RandomForest

Saved model: RandomForest

--- Evaluating saved model on full data ---
Confusion matrix:
[[212   0]
 [  0 357]]

Classification report:
              precision    recall  f1-score   support

   malignant       1.00      1.00      1.00       212
      benign       1.00      1.00      1.00       357

    accuracy                           1.00       569
   macro avg       1.00      1.00      1.00       569
weighted avg       1.00      1.00      1.00       569



In [23]:
%%writefile clf_train.py
from clf_delivery import load_data, get_features_and_target, train_best_and_save

df = load_data()
X, y = get_features_and_target(df)
model, name = train_best_and_save(X, y)
print("\nTraining done. Best model:", name)

Writing clf_train.py


In [36]:
%%writefile clf_predict.py
import sys
import joblib
import pandas as pd
import numpy as np
from clf_delivery import load_data, is_valid_order

radius_mean    = float(sys.argv[1])
area_mean      = float(sys.argv[2])
texture_mean   = float(sys.argv[3])
perimeter_mean = float(sys.argv[4])

if not is_valid_order(radius_mean, area_mean, texture_mean, perimeter_mean):
    print("Invalid order")
    sys.exit(0)

model = joblib.load("clf_model.joblib")
df = load_data()
feature_cols = [c for c in df.columns if c != "target"]


closest_idx = (df["mean radius"] - radius_mean).abs().idxmin()
base = df.loc[closest_idx, feature_cols].to_dict()

base["mean radius"]    = radius_mean
base["mean area"]      = area_mean
base["mean texture"]   = texture_mean
base["mean perimeter"] = perimeter_mean


r_ratio = radius_mean / df.loc[closest_idx, "mean radius"]
base["radius error"]    = df.loc[closest_idx, "radius error"]    * r_ratio
base["perimeter error"] = df.loc[closest_idx, "perimeter error"] * r_ratio
base["area error"]      = df.loc[closest_idx, "area error"]      * (r_ratio ** 2)
base["worst radius"]    = df.loc[closest_idx, "worst radius"]    * r_ratio
base["worst perimeter"] = df.loc[closest_idx, "worst perimeter"] * r_ratio
base["worst area"]      = df.loc[closest_idx, "worst area"]      * (r_ratio ** 2)

order = pd.DataFrame([base])[feature_cols]

pred  = model.predict(order)[0]
proba = model.predict_proba(order)[0]

label      = "benign" if pred == 1 else "malignant"
confidence = proba[pred]

print(f"Prediction : {label}")
print(f"Confidence : {confidence:.2%}")
print(f"Full proba : malignant={proba[0]:.2%}, benign={proba[1]:.2%}")

Overwriting clf_predict.py


In [35]:
!python clf_train.py

LogisticRegression   CV accuracy = 0.9508
DecisionTree         CV accuracy = 0.9209
RandomForest         CV accuracy = 0.9543

Best model: RandomForest

Training done. Best model: RandomForest


In [38]:
!python clf_predict.py 14.5 650 19 92
!python clf_predict.py 13.0 500 18 82
!python clf_predict.py -5 600 20 90

Validation: 4/4 measurements in realistic range
Prediction : benign
Confidence : 94.00%
Full proba : malignant=6.00%, benign=94.00%
Validation: 4/4 measurements in realistic range
Prediction : malignant
Confidence : 92.00%
Full proba : malignant=92.00%, benign=8.00%
Validation: negative/zero value found — rejected
Invalid order
